# Part 1 — Restoring force versus impact position (square lattice)

Uses the exact solver in `surface.py` to compute the z-restoring force for a fixed indentation depth at each impact location, and plots the force map (Figure 3a in the manuscript).

> **Note.** This notebook imports the `surface` module. Keep `surface.py` (provided in this folder) in the same directory, or run this notebook from `part1/`.

In [ ]:
import surface
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import numpy as np

def spring_force(spring,dot1,dot2):
    x1,y1,z1 = dot1
    x2,y2,z2 = dot2
    Fx,Fy,Fz = map(lambda x:spring*x, (x2-x1, y2-y1, z2-z1))
    return [Fx,Fy,Fz]

N = 20
X, Y = 5, 5
DEPTH = 0.5
TOT_DEPTH = 0.8
L = 1
spring = 0.1

def force(N, X, Y, DEPTH, TOT_DEPTH):
    data = surface.surface(N, X, Y, DEPTH, TOT_DEPTH)
    Force = [0,0,0]
    near = [(1,0), (-1,0), (0,1), (0,-1)]
    near_hight = {}
    near_force = {}

    for dir in near:
        index = (X+dir[0], Y+dir[1])
        near_hight[dir] = data[index[1]][index[0]]
        near_force[dir] = spring_force(spring, (X, Y, DEPTH), (index[0], index[1], near_hight[dir]))

    for i in range(3):
        Force[i] = sum(force[i] for force in near_force.values())
    return Force[2]

Force_data = [[0]*(N-2) for _ in range(N-2)]
min_Force = float('inf')
max_Force = float('-inf')
for x in range(1,N-1):
    for y in range(1,N-1):
        Force_data[y-1][x-1] = -force(N, x, y, DEPTH, TOT_DEPTH)
        min_Force = min(min_Force, Force_data[y-1][x-1])
        max_Force = max(max_Force, Force_data[y-1][x-1])
ax = plt.axes(projection='3d')
x = np.linspace(0, N-3, N-2)
y = np.linspace(0, N-3, N-2)
z = ax.set_zlim(min_Force,max_Force)
X,Y = np.meshgrid(x,y)
Z = np.array(Force_data)
ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax.set_title('FORCE')
ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('FORCE')
plt.show()
